# 06 — Train LightGBM LambdaRank

Group = số candidate / user. Label = 1 nếu (u, i) ∈ valid_gt.
Sample weight: view_phone=3, contact_*=2, other_interaction=1 (cho positive).

In [ ]:
# Local kernel: assume deps already installed.
# To install run once:
#   pip install pyarrow pandas numpy scipy lightgbm scikit-learn tqdm
print("Skipping pip install (local kernel).")

In [ ]:
# === Local setup: paths + add training/utils to sys.path ===
import os, sys

PROJECT_DIR  = r"d:/Datathon_2"
TRAINING_DIR = os.path.join(PROJECT_DIR, "training")
CACHE_DIR    = os.path.join(PROJECT_DIR, "cache_drive")
DATA_DIR     = os.path.join(PROJECT_DIR, "Datathon_Data")  # contains train/ and test/
os.makedirs(CACHE_DIR, exist_ok=True)

if TRAINING_DIR not in sys.path:
    sys.path.insert(0, TRAINING_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR   :", DATA_DIR)
print("CACHE_DIR  :", CACHE_DIR)
print("utils available:", os.path.isdir(os.path.join(TRAINING_DIR, "utils")))

In [ ]:
TRAIN_DATE_START = "2025-11-09"
TRAIN_DATE_END = "2026-04-09"
VALID_START = "2026-03-13"

POSITIVE_EVENT_TYPES = frozenset({
    "view_phone", "contact_chat", "other_interaction", "contact_zalo", "contact_sms",
})
HIGH_INTENT_EVENTS = frozenset({"view_phone", "contact_chat", "contact_zalo", "contact_sms"})

INTENT_WEIGHT = {
    "view_phone": 3.0, "contact_chat": 2.0,
    "contact_zalo": 2.0, "contact_sms": 2.0,
    "other_interaction": 1.0,
}

# Local data paths (relative to DATA_DIR defined in the setup cell)
TRAIN_PATH = os.path.join(DATA_DIR, "train") + os.sep
TEST_PATH  = os.path.join(DATA_DIR, "test", "test") + os.sep

print("Constants loaded. TRAIN_END:", TRAIN_DATE_END, "VALID_START:", VALID_START)
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH :", TEST_PATH)

In [ ]:
# Local mode: no GCS egress guardrail needed.
print("Local kernel: reading from CACHE_DIR.")

In [ ]:
import time
import numpy as np
import pandas as pd
import lightgbm as lgb

from utils.metrics import mean_recall_at_k, mean_ndcg_at_k

t0 = time.time()
full = pd.read_parquet(f"{CACHE_DIR}/train_matrix_valid.parquet")
print(f"full: {full.shape} | {time.time()-t0:.1f}s")

# Drop non-feature columns
DROP = {"user_id", "item_id", "label", "title", "posted_date",
        "expected_expired_date", "first_evt_date", "last_evt_date",
        "last_snap_date", "project_id"}
cat_cols = [c for c in full.columns
            if c not in DROP and full[c].dtype == "object"]
num_cols = [c for c in full.columns
            if c not in DROP and full[c].dtype != "object"
            and full[c].dtype != "datetime64[ns]"]
for c in cat_cols:
    full[c] = full[c].astype("category")
feat_cols = cat_cols + num_cols
print(f"#features: {len(feat_cols)} (cat={len(cat_cols)}, num={len(num_cols)})")

In [ ]:
# Split inside the valid users: 80/20 by user_id hash to avoid leakage
from hashlib import md5
def _hash01(s):
    return int(md5(str(s).encode()).hexdigest(), 16) % 100

full["_h"] = full["user_id"].map(_hash01)
tr_mask = full["_h"] < 80
va_mask = full["_h"] >= 80

X_tr = full.loc[tr_mask, feat_cols]
y_tr = full.loc[tr_mask, "label"].values
g_tr = full.loc[tr_mask].groupby("user_id", sort=False).size().values

X_va = full.loc[va_mask, feat_cols]
y_va = full.loc[va_mask, "label"].values
g_va = full.loc[va_mask].groupby("user_id", sort=False).size().values

print(f"train: {len(X_tr):,} ({tr_mask.sum()} rows, "
      f"{len(g_tr)} groups, pos={y_tr.sum()})")
print(f"valid: {len(X_va):,} (pos={y_va.sum()})")

In [ ]:
# Monotonic constraints: force model to respect domain logic regardless of Tet noise
MONO_MAP = {
    "age_at_train_end":         -1,  # older listing -> lower score
    "recency_evt_days":         -1,  # stale listing -> lower score
    "u_recency_days":           -1,  # user inactive longer -> lower score
    "i_CR_30d":                  1,  # higher CVR -> higher score
    "i_contacts_24h_mean_30d":   1,  # more contacts -> higher score
    "i_n_pos_30d":               1,  # more positive events -> higher score
    "i_n_pos_7d":                1,
    "u_n_pos_30d":               1,  # active user -> higher score
    # Super features
    "i_velocity_contacts":       1,  # trending up in contacts -> higher score
    "i_velocity_views":          1,  # trending up in views -> higher score
    "x_price_affinity":          1,  # closer to user price pref -> higher score
}
mono_list = [MONO_MAP.get(c, 0) for c in feat_cols]
print(f"Monotonic constraints: {sum(v!=0 for v in mono_list)} active / {len(mono_list)} features")

params = dict(
    objective="lambdarank", metric="ndcg", ndcg_eval_at=[10],
    learning_rate=0.05, num_leaves=127, min_data_in_leaf=200,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    lambda_l2=1.0, max_bin=255, seed=42, verbosity=-1,
    monotone_constraints=mono_list,
    monotone_constraints_method="advanced",  # best for lambdarank
)
d_tr = lgb.Dataset(X_tr, label=y_tr, group=g_tr,
                   categorical_feature=cat_cols, free_raw_data=False)
d_va = lgb.Dataset(X_va, label=y_va, group=g_va,
                   categorical_feature=cat_cols, reference=d_tr,
                   free_raw_data=False)

t0 = time.time()
model = lgb.train(
    params, d_tr, num_boost_round=2000,
    valid_sets=[d_va], valid_names=["valid"],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(50)],
)
print(f"trained in {time.time()-t0:.0f}s | best iter: {model.best_iteration}")
model.save_model(f"{CACHE_DIR}/model.txt")

fi = pd.DataFrame({"feature": feat_cols,
                    "gain": model.feature_importance(importance_type="gain"),
                    "split": model.feature_importance(importance_type="split")})
fi = fi.sort_values("gain", ascending=False)
fi.to_csv(f"{CACHE_DIR}/feature_importance.csv", index=False)
print(fi.head(20))

In [ ]:
# Internal Recall@10 / NDCG@10 on valid split
full.loc[va_mask, "_pred"] = model.predict(X_va, num_iteration=model.best_iteration)
preds_dict = {}
for uid, grp in full[va_mask].groupby("user_id", sort=False):
    top10 = grp.nlargest(10, "_pred")["item_id"].tolist()
    preds_dict[uid] = top10

valid_gt = pd.read_parquet(f"{CACHE_DIR}/valid_gt.parquet")
gt = valid_gt.groupby("user_id")["item_id"].agg(set).to_dict()
gt_va = {u: gt[u] for u in preds_dict if u in gt}
r10 = mean_recall_at_k(preds_dict, gt_va, k=10)
n10 = mean_ndcg_at_k(preds_dict, gt_va, k=10)
print(f"Internal Recall@10: {r10:.4f}")
print(f"Internal NDCG@10  : {n10:.4f}")

import json
with open(f"{CACHE_DIR}/ranker_metrics.json", "w") as f:
    json.dump({"recall_at_10": r10, "ndcg_at_10": n10,
               "best_iter": int(model.best_iteration)}, f, indent=2)

In [ ]:
# ── SHAP feature selection ──────────────────────────────────────────────────
# Sample 5k rows from valid split (SHAP is slow on full matrix)
import shap, json

SHAP_SAMPLE = 5_000
rng = np.random.default_rng(42)
va_idx = np.where(va_mask.values)[0]
samp_idx = rng.choice(va_idx, size=min(SHAP_SAMPLE, len(va_idx)), replace=False)
X_shap = full.iloc[samp_idx][feat_cols]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_shap)          # shape: (n_samples, n_features)

shap_mean_abs = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({
    "feature": feat_cols,
    "shap_mean_abs": shap_mean_abs,
}).sort_values("shap_mean_abs", ascending=False)
shap_df.to_csv(f"{CACHE_DIR}/shap_importance.csv", index=False)

# Features that contribute near-zero SHAP value are noise candidates
ZERO_THRESHOLD = shap_df["shap_mean_abs"].max() * 0.001
noise_feats = shap_df[shap_df["shap_mean_abs"] < ZERO_THRESHOLD]["feature"].tolist()
print(f"Total features: {len(feat_cols)}")
print(f"Noise features (SHAP < {ZERO_THRESHOLD:.4f}): {len(noise_feats)}")
print(noise_feats[:20])

with open(f"{CACHE_DIR}/shap_noise_features.json", "w") as f:
    json.dump(noise_feats, f, indent=2)

print(shap_df.head(15).to_string(index=False))